In [0]:
table_names = [
    "workspace.default.bronze_facilities",
    "workspace.default.bronze_india_post_pincode_directory",
    "workspace.default.bronze_nfhs_5_district_health_indicators",
]

for table_name in table_names:
    print()
    print(table_name)
    table = spark.table(table_name)
    row_count = table.count()
    columns = table.dtypes

    print(f"Rows: {row_count:,}")
    print(f"Columns: {len(columns)}")
    for column_name, data_type in columns:
        print(f"  {column_name} ({data_type})")




workspace.default.bronze_facilities
Rows: 10,088
Columns: 51
  unique_id (string)
  source_types (string)
  source_ids (string)
  source_content_id (string)
  name (string)
  organization_type (string)
  content_table_id (string)
  phone_numbers (string)
  officialPhone (string)
  email (string)
  websites (string)
  officialWebsite (string)
  yearEstablished (string)
  acceptsVolunteers (string)
  facebookLink (string)
  address_line1 (string)
  address_line2 (string)
  address_line3 (string)
  address_city (string)
  address_stateOrRegion (string)
  address_zipOrPostcode (string)
  address_country (string)
  address_countryCode (string)
  countries (string)
  facilityTypeId (string)
  operatorTypeId (string)
  affiliationTypeIds (string)
  description (string)
  area (string)
  numberDoctors (string)
  capacity (string)
  specialties (string)
  procedure (string)
  equipment (string)
  capability (string)
  recency_of_page_update (string)
  distinct_social_media_presence_count (stri

In [0]:
%sql
SELECT
    CASE WHEN table_catalog = 'workspace' THEN 'BRONZE' ELSE 'SOURCE' END AS side,
    table_catalog,
    table_schema,
    table_name,
    table_type
FROM system.information_schema.tables
WHERE (
    table_catalog = 'databricks_virtue_foundation_dataset_dais_2026'
    AND table_schema <> 'information_schema'
) OR (
    table_catalog = 'workspace'
    AND table_schema = 'default'
    AND startswith(table_name, 'bronze_')
)
ORDER BY table_catalog, table_schema, table_name

side,table_catalog,table_schema,table_name,table_type
BRONZE,workspace,default,bronze_facilities,MANAGED
BRONZE,workspace,default,bronze_india_post_pincode_directory,MANAGED
BRONZE,workspace,default,bronze_nfhs_5_district_health_indicators,MANAGED


In [0]:
from itertools import zip_longest

source_schema = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset"
bronze_schema = "workspace.default"
tables = ["facilities", "india_post_pincode_directory", "nfhs_5_district_health_indicators"]

for name in tables:
    try:
        source = spark.table(f"{source_schema}.{name}")
        bronze = spark.table(f"{bronze_schema}.bronze_{name}")

        source_columns = source.dtypes
        bronze_columns = bronze.dtypes
        if source_columns != bronze_columns:
            pairs = zip_longest(source_columns, bronze_columns)
            for position, (original, copy) in enumerate(pairs, start=1):
                if original != copy:
                    print(f"{name} | SCHEMA MISMATCH | column {position}: {original} vs {copy}")
                    break
            continue

        source_rows = source.count()
        bronze_rows = bronze.count()
        missing = source.exceptAll(bronze).count()
        extra = bronze.exceptAll(source).count()

        match = source_rows == bronze_rows and missing == 0 and extra == 0
        status = "PASS" if match else "DATA MISMATCH"
        print(f"{name} | {status} | source={source_rows:,} bronze={bronze_rows:,} "
              f"missing={missing:,} extra={extra:,}")
    except Exception as error:
        message = str(error).strip() or type(error).__name__
        print(f"{name} | ERROR | {message.splitlines()[0]}")

facilities | ERROR | [INSUFFICIENT_PERMISSIONS] Insufficient privileges:
india_post_pincode_directory | ERROR | [INSUFFICIENT_PERMISSIONS] Insufficient privileges:
nfhs_5_district_health_indicators | ERROR | [INSUFFICIENT_PERMISSIONS] Insufficient privileges:


In [0]:
print(message)

[INSUFFICIENT_PERMISSIONS] Insufficient privileges:
Principal '<user>' does not have USE CATALOG on Catalog 'databricks_virtue_foundation_dataset_dais_2026'. SQLSTATE: 42501

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at com.databricks.sql.managedcatalog.client.TypeConversionUtils.toAnalysisException(TypeConversionUtils.scala:4033)
	at com.databricks.sql.managedcatalog.client.TypeConversionUtils.toAnalysisException$(TypeConversionUtils.scala:4016)
	at com.databricks.managedcatalog.TypeConversionUtils$.toAnalysisException(ManagedCatalogClientShim.scala:18)
	at com.databricks.sql.managedcatalog.client.TypeConversionUtils.toCatalyst(TypeConversionUtils.scala:3576)
	at com.databricks.sql.managedcatalog.client.TypeConversionUtils.toCatalyst$(TypeConversionUtils.scala:3562)
	at com.databricks.managedcatalog.TypeConversionUtils$.toCatalyst(ManagedCatalogClientShim.scala:18)
	at com.databricks.sql.managedcatalog.client.ManagedCatalogClientImpl.$anonfun$bulkGetMetadataWithPermissio

In [0]:
tables = [
    "workspace.default.bronze_facilities",
    "workspace.default.bronze_india_post_pincode_directory",
    "workspace.default.bronze_nfhs_5_district_health_indicators",
]

for table in tables:
    try:
        detail = spark.sql(f"DESCRIBE DETAIL {table}").first()
        if detail["format"].lower() != "delta":
            print(f"{table} | REVIEW | format={detail['format']}")
            continue

        history = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1")
        commit = history.selectExpr(
            "version", "operation",
            "date_format(timestamp, 'yyyy-MM-dd HH:mm:ss XXX') AS committed_at"
        ).first()
        if commit is None:
            print(f"{table} | REVIEW | no commit found")
            continue

        version = commit["version"]
        snapshot = f"{table} VERSION AS OF {version}"
        rows = spark.sql(f"SELECT COUNT(*) AS n FROM {snapshot}").first()["n"]
        print(f"{table} | READY | version={version} | rows={rows:,} | "
              f"committed={commit['committed_at']} | operation={commit['operation']}")
        print(f"SELECT * FROM {snapshot};")
    except Exception as err:
        message = str(err).partition("JVM stacktrace:")[0].strip()
        print(f"{table} | ERROR | {message or type(err).__name__}")

workspace.default.bronze_facilities | READY | version=0 | rows=10,088 | committed=2026-11-21 16:43:42 Z | operation=CREATE OR REPLACE TABLE AS SELECT
SELECT * FROM workspace.default.bronze_facilities VERSION AS OF 0;
workspace.default.bronze_india_post_pincode_directory | READY | version=0 | rows=165,627 | committed=2026-11-21 16:43:48 Z | operation=CREATE OR REPLACE TABLE AS SELECT
SELECT * FROM workspace.default.bronze_india_post_pincode_directory VERSION AS OF 0;
workspace.default.bronze_nfhs_5_district_health_indicators | READY | version=0 | rows=706 | committed=2026-11-21 16:43:53 Z | operation=CREATE OR REPLACE TABLE AS SELECT
SELECT * FROM workspace.default.bronze_nfhs_5_district_health_indicators VERSION AS OF 0;
